# IMDb Sentiment Analysis — Data Preparation
Load the IMDb dataset, inspect batches, build a vocabulary, and test text encoding and padding.

## 1. Imports and dataset

In [11]:
import sys
from pathlib import Path

root_dir = Path().resolve().parent
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))

In [12]:
from torch.utils.data import DataLoader
from src.data.dataset import IMDbDataset
from src.data.tokenizer import *
from src.data.vocabulary import Vocabulary
from src.data.preprocessing import pad_or_truncate

In [13]:
train_dataset = IMDbDataset("../data/aclImdb", train=True)
test_dataset = IMDbDataset("../data/aclImdb", train=False)

print(len(train_dataset))  # 25000
print(len(test_dataset))   # 25000

review, label = train_dataset[0]
print(review)
print(label)

25000
25000
Bromwell High is a cartoon comedy. It ran at the same time as some other programs about school life, such as "Teachers". My 35 years in the teaching profession lead me to believe that Bromwell High's satire is much closer to reality than is "Teachers". The scramble to survive financially, the insightful students who can see right through their pathetic teachers' pomp, the pettiness of the whole situation, all remind me of the schools I knew and their students. When I saw the episode in which a student repeatedly tried to burn down the school, I immediately recalled ......... at .......... High. A classic line: INSPECTOR: I'm here to sack one of your teachers. STUDENT: Welcome to Bromwell High. I expect that many adults of my age think that Bromwell High is far fetched. What a pity that it isn't!
1


## 2. Batching with DataLoader
The default `DataLoader` collates labels into tensors, while raw review strings remain strings.

In [14]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

train_reviews, train_labels = next(iter(train_loader))
print(type(train_reviews))  # tuple of reviews
print(type(train_labels))   # torch.Tensor
print(len(train_reviews))   # 32
print(train_labels.shape)   # torch.Size([32])
print(train_reviews[0])

<class 'tuple'>
<class 'torch.Tensor'>
32
torch.Size([32])
This is by far the worst movie I've ever seen. From the plot, though the shots, the "special effects", the acting, and did I mentioned the plot? Every single thing in it sucked ass!<br /><br />This is a good example of what "over-doing" means and I'll try to explain:<br /><br />I understand what the creator of this movie was trying to do; this was supposed to be one of those movies you can't really tell what the hell is going on up until the end. You sit mesmerized, not knowing who's dead and who's alive and who killed who and why and just when you think you got the timeline right you find out you're wrong and the movie ends - leaving you with an opened mouth for the next 5 minutes! Something like "Unbreakable" or "The Job" if you know what I mean.<br /><br />But Olga Levens, the writer of this junk, yes, Olga  The writer, Director, Producer, Screenplay author, Caster, Production designer, Art Director, Costume Designer and al

## 3. Build the vocabulary
Build it using training reviews only, so information from the test set does not leak into preprocessing.

In [15]:
tokenizer = WhitespaceTokenizer()
vocab = Vocabulary(tokenizer, min_freq=2)
vocab.build([review for review, _ in train_dataset])  # Build vocabulary from training reviews

In [16]:
text = "this movie is great"
print(f"Original text: {text}")

encoded_text = vocab.encode(text)
print(f"Encoded text: {encoded_text}")

decoded_text = vocab.decode(encoded_text)
print(f"Decoded text: {decoded_text}")

Original text: this movie is great
Encoded text: [305, 383, 4, 1282]
Decoded text: this movie is great


## 4. Padding and truncation
Convert a review into token IDs first, then pad or truncate that sequence to the chosen length.

In [18]:
padded_sequence = pad_or_truncate(
    encoded_text,
    max_length=10,
    pad_id=vocab.PAD_ID
)
print(f"Processed sequence: {padded_sequence}")
print(f"Decoded padded sequence: {vocab.decode(padded_sequence)}")

Processed sequence: [305, 383, 4, 1282, 0, 0, 0, 0, 0, 0]
Decoded padded sequence: this movie is great <PAD> <PAD> <PAD> <PAD> <PAD> <PAD>


In [19]:
encoded_review = vocab.encode(train_reviews[0])
padded_review = pad_or_truncate(
    encoded_review,
    max_length=100,
    pad_id=vocab.PAD_ID
)

print(f"Padded sequence of first training review: {padded_review}")
print(f"Decoded padded sequence of first training review: {vocab.decode(padded_review)}")
print(f"Length of processed review: {len(padded_review)}")

Padded sequence of first training review: [382, 4, 219, 95, 11, 2219, 383, 1705, 319, 2117, 5146, 11, 265, 536, 11, 2048, 11, 19836, 1, 11, 2255, 62, 120, 60, 5240, 11, 1664, 3471, 2233, 920, 26, 99, 7643, 1, 153, 982, 4, 5, 329, 7405, 54, 155, 1, 2400, 62, 1626, 2229, 31, 1, 153, 531, 2095, 155, 11, 19846, 54, 305, 383, 449, 3584, 31, 81766, 305, 449, 1605, 31, 148, 85, 54, 113, 451, 156, 212, 662, 1310, 155, 11, 2996, 4, 123, 114, 151, 187, 11, 1066, 951, 2199, 21929, 249, 4326, 6028, 8315, 62, 6028, 1982, 62, 45, 2061, 45, 62]
Decoded padded sequence of first training review: This is by far the worst movie I've ever seen. From the plot, though the shots, the "special <UNK> the acting, and did I mentioned the plot? Every single thing in it sucked <UNK> /><br />This is a good example of what <UNK> means and I'll try to <UNK> /><br />I understand what the creator of this movie was trying to do; this was supposed to be one of those movies you can't really tell what the hell is going on 